In [12]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
print(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique values in inbound:")
print(df["inbound"].unique())

print("\nCount of each inbound value:")
print(df["inbound"].value_counts())

Shape: (2811774, 7)

Columns:
Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

First 5 rows:
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                

In [19]:
# Calculate author-level statistics

author_stats = df.groupby("author_id").agg(
    inbound_tweets=("inbound", "sum"),
    total_tweets=("tweet_id", "count")
)

author_stats["outbound_tweets"] = (
    author_stats["total_tweets"] -
    author_stats["inbound_tweets"]
)

author_stats = author_stats[
    ["inbound_tweets", "outbound_tweets", "total_tweets"]
]

print("\nAuthor-level statistics:")
print(author_stats.head())

# Authors with at least one outbound tweet
authors_with_outbound = author_stats[
    author_stats["outbound_tweets"] > 0
]

print("\nAuthors with at least one outbound tweet:")
print(authors_with_outbound.shape)

# Top authors by outbound tweet count
top_outbound = authors_with_outbound.sort_values(
    "outbound_tweets",
    ascending=False
)

print(top_outbound.head(20))


Author-level statistics:
           inbound_tweets  outbound_tweets  total_tweets
author_id                                               
10026                   3                0             3
100363                  1                0             1
10103                   2                0             2
10221                   2                0             2
10286                   1                0             1

Authors with at least one outbound tweet:
(108, 3)
                 inbound_tweets  outbound_tweets  total_tweets
author_id                                                     
AmazonHelp                    0           169840        169840
AppleSupport                  0           106860        106860
Uber_Support                  0            56270         56270
SpotifyCares                  0            43265         43265
Delta                         0            42253         42253
Tesco                         0            38573         38573
AmericanAir        

In [20]:
# Select only outbound tweets
outbound = df[df["inbound"] == False]

# Connect each outbound tweet to the tweet it is responding to
outbound_customer = outbound.merge(
    df[["tweet_id", "author_id"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_brand", "_customer")
)

# Count distinct customers for each brand
customer_counts = (
    outbound_customer
    .groupby("author_id_brand")["author_id_customer"]
    .nunique()
    .reset_index(name="distinct_customers")
)

# Sort by number of customers
customer_counts = customer_counts.sort_values(
    "distinct_customers",
    ascending=False
)

# Display top authors
customer_counts.head(20)

,author_id_brand,distinct_customers
10,AppleSupport,76366
8,AmazonHelp,71049
85,Uber_Support,38300
77,SpotifyCares,27794
40,Delta,22331
99,comcastcares,21824
9,AmericanAir,21686
78,TMobileHelp,19943
76,SouthwestAir,19713
26,Ask_Spectrum,17214


In [ ]:
# Select AppleSupport's outbound tweets
apple_outbound = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False)
]

# Get the customer tweet IDs that AppleSupport responded to
apple_customer_tweets = apple_outbound[
    "in_response_to_tweet_id"
].dropna()

# Count unique customer tweets
number_of_conversations = apple_customer_tweets.nunique()

print("Number of conversation threads involving AppleSupport:",
      number_of_conversations)

Number of conversation threads involving AppleSupport: 106696


In [23]:
# Convert parent tweet ID to numeric

df["in_response_to_tweet_id"] = pd.to_numeric(
    df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

# Create mapping:
#    tweet_id → tweet id is replying to

parent = dict(
    zip(
        df["tweet_id"],
        df["in_response_to_tweet_id"]
    )
)

# Function to find the original/root tweet

def find_root(tweet_id):
    visited = set()

    while pd.notna(parent.get(tweet_id)):

        # Prevent infinite loops
        if tweet_id in visited:
            break

        visited.add(tweet_id)

        tweet_id = int(parent[tweet_id])

    return tweet_id

# Assign conversation ID to every tweet

df["conversation_id"] = df["tweet_id"].map(find_root)

# Define the 5 brands

brands = [
    "AppleSupport",
    "AmazonHelp",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

# Calculate conversation statistics

results = []

for brand in brands:

    # Tweets posted by this brand
    brand_tweets = df[
        df["author_id"] == brand
    ]

    # Conversation IDs involving this brand
    conversation_ids = brand_tweets[
        "conversation_id"
    ].unique()

    # Get all tweets belonging to those conversations
    conversations = df[
        df["conversation_id"].isin(conversation_ids)
    ]

    # Number of complete conversation threads
    number_of_conversations = (
        conversations["conversation_id"].nunique()
    )

    # Number of customer messages
    number_of_customer_messages = (
        conversations["inbound"] == True
    ).sum()

    # Number of brand responses
    number_of_brand_responses = (
        (conversations["author_id"] == brand) &
        (conversations["inbound"] == False)
    ).sum()

    # Store results
    results.append({
        "Brand": brand,
        "Number of conversations": number_of_conversations,
        "Number of customer messages": number_of_customer_messages,
        "Number of brand responses": number_of_brand_responses
    })


conversation_summary = pd.DataFrame(results)

print(conversation_summary)

          Brand  Number of conversations  Number of customer messages  \
0  AppleSupport                    80702                       131764   
1    AmazonHelp                    82534                       203598   
2  Uber_Support                    41923                        72154   
3  SpotifyCares                    28280                        48543   
4         Delta                    26166                        45296   

   Number of brand responses  
0                     106860  
1                     169840  
2                      56270  
3                      43265  
4                      42253  


In [32]:
# ============================================================
# Validate reconstructed AppleSupport conversations
# ============================================================

# 1. Get all conversation IDs involving AppleSupport
apple_conversation_ids = df[
    df["author_id"] == "AppleSupport"
]["conversation_id"].dropna().unique()


# 2. Select 5 conversation IDs
sample_conversations = pd.Series(
    apple_conversation_ids
).sample(30, random_state=42)


# 3. Display each conversation
for conversation_id in sample_conversations:

    print("=" * 100)
    print(f"Conversation: {conversation_id}")
    print("=" * 100)

    # Get all tweets belonging to this conversation
    conversation = df[
        df["conversation_id"] == conversation_id
    ].copy()

    # Sort chronologically
    conversation = conversation.sort_values("created_at")

    # Display required columns
    print(
        conversation[
            [
                "conversation_id",
                "tweet_id",
                "created_at",
                "author_id",
                "inbound",
                "text"
            ]
        ].to_string(index=False)
    )

    print("\n")

Conversation: 1216340
 conversation_id  tweet_id                     created_at    author_id  inbound                                                                                                                                                           text
         1216340   1216340 Wed Oct 25 17:02:51 +0000 2017       405412     True                                   @AppleSupport hi, my gifs are sending extremely small to others but I receive their gifs at normal size. How can I fix this?
         1216340   1216338 Wed Oct 25 17:21:29 +0000 2017 AppleSupport    False                                                @405412 We're happy to take a look in to this with you and figure out what is going on! What device is this on?
         1216340   1216339 Wed Oct 25 17:23:26 +0000 2017       405412     True @AppleSupport Thank you! It’s on an iPhone7 Plus running on iOS 11.0.3. So far I’ve restarted my phone &amp; also toggled iMessage on and off but no luck. Lol
         1216340   121

In [26]:
conversation_summary["Average messages per conversation"] = (
    conversation_summary["Number of customer messages"]
    + conversation_summary["Number of brand responses"]
) / conversation_summary["Number of conversations"]

conversation_summary

,Brand,Number of conversations,Number of customer messages,Number of brand responses,Average messages per conversation
0,AppleSupport,80702,131764,106860,2.956854
1,AmazonHelp,82534,203598,169840,4.524657
2,Uber_Support,41923,72154,56270,3.063330
3,SpotifyCares,28280,48543,43265,3.246393
4,Delta,26166,45296,42253,3.345907


In [31]:
# ============================================================
# Validate reconstructed AmazonHelp conversations
# ============================================================

# 1. Get all conversation IDs involving AmazonHelp
amazon_conversation_ids = df[
    df["author_id"] == "AmazonHelp"
]["conversation_id"].dropna().unique()


# 2. Select 5 conversation IDs
sample_conversations = pd.Series(
    amazon_conversation_ids
).sample(30, random_state=42)


# 3. Display each conversation
for conversation_id in sample_conversations:

    print("=" * 100)
    print(f"Conversation: {conversation_id}")
    print("=" * 100)

    # Get all tweets belonging to this conversation
    conversation = df[
        df["conversation_id"] == conversation_id
    ].copy()

    # Sort chronologically
    conversation = conversation.sort_values("created_at")

    # Display required columns
    print(
        conversation[
            [
                "conversation_id",
                "tweet_id",
                "created_at",
                "author_id",
                "inbound",
                "text"
            ]
        ].to_string(index=False)
    )

    print("\n")

Conversation: 2116198
 conversation_id  tweet_id                     created_at  author_id  inbound                                                                                                    text
         2116198   2116198 Wed Nov 08 13:33:51 +0000 2017     623804     True                   amazonミュージック普段あんまり使ってないけど、ペルソナのサントラが聞き放題だときいてみてみたらほんとに凄い数あってびっくりした\n永遠にBGMにこまらないやつだ……
         2116198   2116197 Wed Nov 08 13:43:30 +0000 2017 AmazonHelp    False @623804 Amazon Musicをご利用いただき、ありがとうございます！いろいろなジャンルの曲を取り揃えておりますので、どうぞ存分にお楽しみください♪ (エンドレスで…)O o｡.(´-`*) RI


Conversation: 1402844
 conversation_id  tweet_id                     created_at  author_id  inbound                                                                                                                                         text
         1402844   1402844 Sat Oct 28 17:43:26 +0000 2017     446166     True                       Thanks @115821 for messing up my order just because I deleted an item that hadn't even sh